# Section 1: Make your first Mistral call return data you can trust

Goal: Dock reads 20 customer emails and returns, for each one, the intent, the shipment and the
language, as typed data that your code can check and score.

## Step 1. Look at the inbox

In [1]:
import json
from collections import Counter

inbox = json.loads(open("dock/inbox.json", encoding="utf-8").read())
print(len(inbox), "emails")
print(Counter(mail["from"].split("@")[1] for mail in inbox).most_common(3))
print()
print(json.dumps(inbox[1], indent=1, ensure_ascii=False))

20 emails
[('brightwater.example.com', 4), ('nordlicht.example.com', 3), ('vdberg-bikes.example.com', 3)]

{
 "id": "M02",
 "from": "import@nordlicht.example.com",
 "subject": "Sendung HF-26038 im Zoll",
 "body": "Guten Tag, unsere Sendung HF-26038 steht seit Tagen im Zoll in Hamburg. Welche Dokumente fehlen noch? Wir brauchen die Ware dringend. Mit freundlichen Grüßen, Jonas Weber"
}


## Step 2. Connect to Mistral

In [2]:
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

load_dotenv()
retry = RetryConfig("backoff", BackoffStrategy(1000, 20000, 2.0, 120000),
                    retry_connection_errors=True)
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"], retry_config=retry,
                 timeout_ms=120000)
MODEL = "mistral-small-latest"
print("key loaded:", len(os.environ["MISTRAL_API_KEY"]) > 0)

key loaded: True


## Step 3. A first call

In [3]:
mail = inbox[1]
response = client.chat.complete(
    model=MODEL,
    messages=[{"role": "user",
               "content": f"What does this customer want?\n\n{mail['body']}"}],
)
print(response.choices[0].message.content)
print()
print(response.usage.prompt_tokens, "tokens in,", response.usage.completion_tokens, "out")

The customer, Jonas Weber, is urgently requesting information about the missing documents for their shipment **HF-26038**, which has been held at customs in Hamburg for several days. They need the goods urgently and are asking:

1. **What documents are still missing?**
2. **How to resolve the delay?**

The tone is formal but urgent, indicating time sensitivity. The customer likely expects a quick response with clear next steps to release the shipment.

70 tokens in, 94 out


## Step 4. Ask for one word

In [4]:
LABELS = "tracking, customs, invoice, booking, damage, other"
SYSTEM = f"Classify the customer email. Answer with one word from: {LABELS}."

def label(mail):
    r = client.chat.complete(model=MODEL, messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": mail["subject"] + "\n\n" + mail["body"]}])
    return r.choices[0].message.content

answers = {mail["id"]: label(mail) for mail in inbox}
print(Counter(answers.values()))

Counter({'customs': 5, 'tracking': 4, 'invoice': 3, 'booking': 3, 'damage': 3, 'other': 2})


In [5]:
allowed = set(LABELS.split(", "))
odd = {k: v for k, v in answers.items() if v not in allowed}
print(len(odd), "answers your code can't use:", odd)

0 answers your code can't use: {}


## Step 5. A schema instead of a sentence

In [6]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class Triage(BaseModel):
    intent: Literal["tracking", "customs", "invoice", "booking", "damage", "other"]
    shipment_id: Optional[str] = Field(
        None, description="Harbor Freight shipment ref like HF-26001, or null if none")
    language: str = Field(description="ISO 639-1 code of the email, like en or de")


def triage(mail):
    r = client.chat.parse(model=MODEL, response_format=Triage, messages=[
        {"role": "system", "content": "You triage emails for Harbor Freight Co."},
        {"role": "user", "content": mail["subject"] + "\n\n" + mail["body"]}])
    return r.choices[0].message.parsed

print(triage(inbox[1]))

intent='customs' shipment_id=None language='de'


In [7]:
results = {mail["id"]: triage(mail) for mail in inbox}
for mid, t in results.items():
    print(mid, f"{t.intent:9} {str(t.shipment_id):9} {t.language}")

M01 tracking  None      en
M02 customs   None      de
M03 invoice   None      en
M04 booking   None      en
M05 damage    None      en
M06 other     None      en
M07 tracking  None      nl-NL
M08 customs   None      en
M09 booking   None      en
M10 tracking  None      en
M11 invoice   None      fr
M12 customs   None      en
M13 damage    None      en
M14 booking   None      de
M15 tracking  None      en
M16 other     None      en
M17 invoice   None      en
M18 damage    None      en
M19 tracking  None      en
M20 customs   None      en


## Step 6. A better schema

In [8]:
class TriageV2(BaseModel):
    intent: Literal["tracking", "customs", "invoice", "booking", "damage", "other"]
    shipment_id: Optional[str] = Field(
        description="The shipment ref exactly as written in the email, like HF-26001. "
                    "null only if the email mentions no shipment")
    language: str = Field(description="Two-letter ISO 639-1 code, lowercase: en, de, fr, nl")


def triage(mail, schema=TriageV2):
    r = client.chat.parse(model=MODEL, response_format=schema, messages=[
        {"role": "system", "content": "You triage emails for Harbor Freight Co."},
        {"role": "user", "content": mail["subject"] + "\n\n" + mail["body"]}])
    return r.choices[0].message.parsed

results = {mail["id"]: triage(mail) for mail in inbox}
for mid, t in results.items():
    print(mid, f"{t.intent:9} {str(t.shipment_id):9} {t.language}")

M01 tracking  HF-26012  en
M02 customs   HF-26038  de
M03 invoice   HF-26040  en
M04 booking   None      en
M05 damage    HF-26037  en
M06 other     None      en
M07 tracking  HF26019   nl
M08 customs   HF-26038  en
M09 booking   None      en
M10 tracking  HF-26016  en
M11 invoice   HF-26031  fr
M12 customs   None      en
M13 damage    HF-26036  en
M14 booking   None      de
M15 tracking  HF-26099  en
M16 other     None      en
M17 invoice   HF-26003  en
M18 damage    HF-26034  en
M19 tracking  HF-26025  en
M20 customs   HF-26033  en


## Step 7. Valid JSON is not correct data

In [9]:
import sqlite3

db = sqlite3.connect("dock/dock.sqlite")

def known(shipment_id):
    row = db.execute("SELECT 1 FROM shipments WHERE id = ?", (shipment_id,))
    return row.fetchone() is not None

for mid, t in results.items():
    if t.shipment_id and not known(t.shipment_id):
        print(mid, repr(t.shipment_id), "is not in the shipments table")

M07 'HF26019' is not in the shipments table
M15 'HF-26099' is not in the shipments table


In [10]:
import re
from pydantic import field_validator

class CheckedTriage(TriageV2):
    @field_validator("shipment_id")
    @classmethod
    def normalize_ref(cls, value):
        if value is None:
            return None
        digits = re.sub(r"\D", "", value)
        return f"HF-{digits}" if len(digits) == 5 else value

    @field_validator("language")
    @classmethod
    def two_letters(cls, value):
        return value.lower()[:2]

fixed = {mid: CheckedTriage(**t.model_dump()) for mid, t in results.items()}
for mid, t in fixed.items():
    if t.shipment_id and not known(t.shipment_id):
        print(mid, repr(t.shipment_id), "still unknown -> ask the customer")

M15 'HF-26099' still unknown -> ask the customer


## Step 8. Score it

In [11]:
truth = json.loads(open("dock/inbox_truth.json").read())
score = Counter()
for mid, t in fixed.items():
    for field in ["intent", "shipment_id", "language"]:
        ok = getattr(t, field) == truth[mid][field]
        score[field] += ok
        if not ok:
            print(mid, field, repr(getattr(t, field)), "expected", repr(truth[mid][field]))
print()
print({field: f"{n}/20" for field, n in score.items()})


{'intent': '20/20', 'shipment_id': '20/20', 'language': '20/20'}


## Step 9. When the call fails

In [12]:
impatient = Mistral(api_key=os.environ["MISTRAL_API_KEY"], timeout_ms=50)
try:
    impatient.chat.complete(model=MODEL, messages=[{"role": "user", "content": "hi"}])
except Exception as error:
    print(type(error).__name__, "-", error)

ReadTimeout - The read operation timed out


In [13]:
tokens = 0
for mail in inbox:
    r = client.chat.parse(model=MODEL, response_format=TriageV2, messages=[
        {"role": "system", "content": "You triage emails for Harbor Freight Co."},
        {"role": "user", "content": mail["subject"] + "\n\n" + mail["body"]}])
    tokens += r.usage.total_tokens
print("20 emails:", tokens, "tokens,", tokens // 20, "per email")

20 emails: 1762 tokens, 88 per email
